# Part d
### Testing different activation functions

In [2]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")



import autograd.numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Code.ffnn2 import NeuralNetwork as FFNN 
from Code.scheduler import Adam, RMS_prop, Constant, Scheduler
from Code.data import runge_function, make_data
from Code.cost import CostOLS, dCostOLS
from Code.comparison import run_comparison
from Code.plot import plot_sweep_heatmap, plot_activation_sweep_heatmap
from Code.activations import sigmoid, RELU, LRELU, identity, derivate
from Code.scheduler import Constant, RMS_prop, Adam
from Code.architectures import build_architectures, build_architectures_exact
from Code.helpers import build_nn, make_sched, train_eval_once, sweep, top_k, best_per
%matplotlib inline


import pandas as pd
import seaborn as sns 
sns.set_theme(style="white", font_scale=1.3)
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 200
plt.rcParams.update({
    "savefig.dpi": 200,
    "savefig.bbox": "tight"
})

rho_val, rho2_val = 0.9, 0.999
optimizers_to_sweep = {
    'GD':       (Constant, {}),
    'SGD':      (Constant, {}),      
    'RMS_prop': (RMS_prop, {'rho': rho_val}),
    'Adam':     (Adam,     {'rho': rho_val, 'rho2': rho2_val}),
}


In [3]:
#Initialize data

(seed_fixed,
 rng_fixed,
 X_fixed_test,
 y_fixed_test,
 N,
 rng,
 x,
 noise,
 y,
 X_train,
 X_val,
 y_train,
 y_val,
 scaler,
 X_train_scaled,
 X_val_scaled,
 X_fixed_test_scaled) = make_data(seed_fixed=42, N=200, noise_std=0.1, test_N=2000)



In [4]:


input_nodes = X_train_scaled.shape[1] # Features from scaled data
output_nodes = 1
layer_output_sizes_2H = [50, 100, output_nodes]
n_hidden_layers = len(layer_output_sizes_2H) - 1  # Should be 2 hidden layers
eta_constant = 1e-2 # if fixed


activation_tests = {
    'Sigmoid': sigmoid,
    'RELU': RELU,
    'LRELU': LRELU
}

epochs_sweep = 100 #keep low for speed, retrain later with top k performers 
batches = 128
lam = 0.0
eta_vals = np.logspace(-4, -1, 4)  # less values than before for speed (removed the worst performing ones)


architectures_to_sweep = build_architectures_exact(out_dim=output_nodes)

## Rough sweep over hyperparameters, activations, architecture, eta, gather the best MSE results

In [5]:

try:
    eta_values = list(eta_constant)   # iterate over several values
except TypeError:
    eta_values = [eta_constant]       # single value

all_activation_results = {}
all_architectural_results = {}
all_results = {}

for arch_name, layer_output_sizes in architectures_to_sweep.items():
    n_hidden_layers = len(layer_output_sizes) - 1
    print(f"\n================ Architecture: {arch_name} ================")
    arch_bucket = {}

    for act_name, h_func in activation_tests.items():
        # hidden activations + identity output (regression)
        activation_funcs = [h_func]*n_hidden_layers + [identity]
        activation_ders  = [derivate(f) for f in activation_funcs]

        act_bucket = {}

        for opt_name, (optimizer_class, fixed_params) in optimizers_to_sweep.items():
            per_eta = {}

            for eta in eta_vals:
                current_kwargs = {'eta': eta}
                current_kwargs.update(fixed_params)

                
                batches_arg = 1 if opt_name == 'GD' else batches

                # train
                nn = FFNN(
                    network_input_size=input_nodes,
                    layer_output_sizes=tuple(layer_output_sizes),
                    activation_funcs=activation_funcs,
                    activation_ders=activation_ders,
                    cost_fun=CostOLS,
                    cost_der=dCostOLS,
                    seed=42
                )
                nn.reset_weights()

                scheduler_instance = optimizer_class(**current_kwargs)

                scores = nn.fit(
                    X_train_scaled, y_train,
                    scheduler=scheduler_instance,
                    batches=batches_arg,               
                    epochs=epochs_sweep,
                    lam=lam
                )

                # Evaluate on the fixed test set
                y_pred = nn.predict(X_fixed_test_scaled)
                test_mse = CostOLS(y_pred.ravel(), y_fixed_test.ravel())

                # Depth/width summary for the print
                depth = max(0, len(layer_output_sizes) - 1)
                width = (layer_output_sizes[0] if depth > 0 else 0)

                print(f" {act_name} + {opt_name} | eta={eta:.1e} | depth={depth}, width={width}: Test MSE={test_mse:.6f}")
                per_eta[float(eta)] = float(test_mse)

            # If only one eta was provided, keep your original shape: opt -> mse
            # If multiple etas, store a dict: opt -> {eta -> mse}
            # Always store per-eta dict, even if it has length 1
            act_bucket[opt_name] = per_eta

        # store results per activation
        arch_bucket[act_name] = act_bucket

    # store results per architecture
    all_results[arch_name] = arch_bucket 
    all_architectural_results[arch_name] = arch_bucket 
    # acts = nn._feed_forward_saver(X_train_scaled)[0] 




================ Architecture: 0_Hidden_Layers ================
 Sigmoid + GD | eta=1.0e-04 | depth=0, width=0: Test MSE=0.167003
 Sigmoid + GD | eta=1.0e-03 | depth=0, width=0: Test MSE=0.146659
 Sigmoid + GD | eta=1.0e-02 | depth=0, width=0: Test MSE=0.097361
 Sigmoid + GD | eta=1.0e-01 | depth=0, width=0: Test MSE=0.094747
 Sigmoid + SGD | eta=1.0e-04 | depth=0, width=0: Test MSE=0.101398
 Sigmoid + SGD | eta=1.0e-03 | depth=0, width=0: Test MSE=0.094750
 Sigmoid + SGD | eta=1.0e-02 | depth=0, width=0: Test MSE=0.095434
 Sigmoid + SGD | eta=1.0e-01 | depth=0, width=0: Test MSE=0.096421
 Sigmoid + RMS_prop | eta=1.0e-04 | depth=0, width=0: Test MSE=0.096012
 Sigmoid + RMS_prop | eta=1.0e-03 | depth=0, width=0: Test MSE=0.095160
 Sigmoid + RMS_prop | eta=1.0e-02 | depth=0, width=0: Test MSE=0.097823
 Sigmoid + RMS_prop | eta=1.0e-01 | depth=0, width=0: Test MSE=0.111308
 Sigmoid + Adam | eta=1.0e-04 | depth=0, width=0: Test MSE=0.108946
 Sigmoid + Adam | eta=1.0e-03 | depth=0, width=

/Users/beateovland/Documents/Fys-stk4155/FYS-STK4155-Project-2/venv/lib/python3.13/site-packages/autograd/numpy/numpy_vjps.py:125: RuntimeWarning: overflow encountered in square
  lambda ans, x, y: unbroadcast_f(y, lambda g: -g * x / y**2),
/Users/beateovland/Documents/Fys-stk4155/FYS-STK4155-Project-2/venv/lib/python3.13/site-packages/autograd/tracer.py:54: RuntimeWarning: overflow encountered in exp
  return f_raw(*args, **kwargs)
/Users/beateovland/Documents/Fys-stk4155/FYS-STK4155-Project-2/venv/lib/python3.13/site-packages/autograd/numpy/numpy_vjps.py:160: RuntimeWarning: invalid value encountered in multiply
  defvjp(anp.exp, lambda ans, x: lambda g: ans * g)


 Sigmoid + SGD | eta=1.0e-01 | depth=1, width=100: Test MSE=nan
 Sigmoid + RMS_prop | eta=1.0e-04 | depth=1, width=100: Test MSE=0.094602
 Sigmoid + RMS_prop | eta=1.0e-03 | depth=1, width=100: Test MSE=0.100625
 Sigmoid + RMS_prop | eta=1.0e-02 | depth=1, width=100: Test MSE=0.094430
 Sigmoid + RMS_prop | eta=1.0e-01 | depth=1, width=100: Test MSE=0.123721
 Sigmoid + Adam | eta=1.0e-04 | depth=1, width=100: Test MSE=0.094285
 Sigmoid + Adam | eta=1.0e-03 | depth=1, width=100: Test MSE=0.095130
 Sigmoid + Adam | eta=1.0e-02 | depth=1, width=100: Test MSE=0.022198
 Sigmoid + Adam | eta=1.0e-01 | depth=1, width=100: Test MSE=0.033559
 RELU + GD | eta=1.0e-04 | depth=1, width=100: Test MSE=0.167068
 RELU + GD | eta=1.0e-03 | depth=1, width=100: Test MSE=0.146614
 RELU + GD | eta=1.0e-02 | depth=1, width=100: Test MSE=0.096790
 RELU + GD | eta=1.0e-01 | depth=1, width=100: Test MSE=0.090351
 RELU + SGD | eta=1.0e-04 | depth=1, width=100: Test MSE=0.100959
 RELU + SGD | eta=1.0e-03 | depth=

## Find best result

Top k runs (ranked by lowest MSE): 

In [ ]:

eta_default = eta_constant  # or e.g. 1e-3

rows = []

#flatten


for arch_name, act_dict in all_results.items():
    for act_name, opt_dict in act_dict.items():
        for opt_name, eta_obj in opt_dict.items():
            # normalize: allow either {eta: mse, ...} or single mse (float)
            if isinstance(eta_obj, dict):
                items = eta_obj.items()
            else:
                if eta_default is None:
                    raise ValueError(
                        f"found scalar MSE for ({arch_name}, {act_name}, {opt_name}) "
                        "but eta_default is None. set eta_default to the eta you used."
                    )
                items = [(eta_default, eta_obj)]

            for eta, mse in items:
                rows.append({
                    "arch_name": arch_name,
                    "act": act_name,
                    "opt": opt_name,
                    "eta": float(eta),
                    "MSE": float(mse),
                })

results_df = (
    pd.DataFrame(rows)
      .sort_values(["arch_name", "act", "opt", "eta"])
      .reset_index(drop=True)
)

k = 10
top_k = results_df.nsmallest(k, "MSE").reset_index(drop=True)
print("top-k configs from sweep:")
print(top_k)


top-k configs from sweep:
                        arch_name      act       opt    eta       MSE
0       2_Hidden_Layers (50, 100)     RELU  RMS_prop  0.001  0.010393
1       2_Hidden_Layers (50, 100)    LRELU  RMS_prop  0.001  0.010440
2       2_Hidden_Layers (50, 100)    LRELU      Adam  0.001  0.010531
3       2_Hidden_Layers (50, 100)     RELU       SGD  0.100  0.010535
4       2_Hidden_Layers (50, 100)    LRELU       SGD  0.100  0.010539
5       2_Hidden_Layers (50, 100)     RELU      Adam  0.001  0.010720
6             1_Hidden_Layer (50)  Sigmoid  RMS_prop  0.010  0.010756
7             1_Hidden_Layer (50)  Sigmoid      Adam  0.100  0.010765
8  3_Hidden_Layers (50, 100, 200)    LRELU      Adam  0.001  0.010843
9       2_Hidden_Layers (50, 100)    LRELU  RMS_prop  0.010  0.010846


In [41]:
import random
def retrain_top_k(
    top_k_df,
    epochs_retrain,
    X_train_scaled,
    y_train,
    X_fixed_test_scaled,
    y_fixed_test,
    lam
):
    retrained = []
    input_nodes = X_train_scaled.shape[1]

    for i, (_, r) in enumerate(top_k_df.iterrows()):
        # Stable, per-config seed:
        seed = 42 + i

        # Force all RNGs to a known state
        np.random.seed(seed)
        random.seed(seed)

        arch_name = r["arch_name"]
        act_name  = r["act"]
        opt_name  = r["opt"]
        eta       = float(r["eta"])

        # Architecture
        layer_output_sizes = architectures_to_sweep[arch_name]
        n_hidden_layers = len(layer_output_sizes) - 1

        # Activation
        h_func = activation_tests[act_name]
        activation_funcs = [h_func]*n_hidden_layers + [identity]
        activation_ders  = [derivate(f) for f in activation_funcs]

        # Optimizer
        optimizer_class, fixed_params = optimizers_to_sweep[opt_name]
        current_kwargs = {"eta": eta}
        current_kwargs.update(fixed_params)

        batches_arg = 1 if opt_name == "GD" else batches

        # Model (also seeded)
        nn = FFNN(
            network_input_size=input_nodes,
            layer_output_sizes=tuple(layer_output_sizes),
            activation_funcs=activation_funcs,
            activation_ders=activation_ders,
            cost_fun=CostOLS,
            cost_der=dCostOLS,
            seed=seed,
        )
        nn.reset_weights()

        scheduler_instance = optimizer_class(**current_kwargs)

        nn.fit(
            X_train_scaled,
            y_train,
            scheduler=scheduler_instance,
            batches=batches_arg,
            epochs=epochs_retrain,
            lam=lam,
        )

        y_pred = nn.predict(X_fixed_test_scaled)
        test_mse = CostOLS(y_pred.ravel(), y_fixed_test.ravel())

        retrained.append({
            "arch_name": arch_name,
            "act": act_name,
            "opt": opt_name,
            "eta": eta,
            "MSE_original": r["MSE"],
            "MSE_retrained": float(test_mse),
        })

    return (
        pd.DataFrame(retrained)
        .sort_values("MSE_retrained", ascending=True)
        .reset_index(drop=True)
    )


In [42]:
epochs_retrain = 500  

retrained_top_k = retrain_top_k(
    top_k,
    epochs_retrain,
    X_train_scaled,
    y_train,
    X_fixed_test_scaled,
    y_fixed_test,
    lam
)

retrained_top_k = (
    retrained_top_k
    .sort_values("MSE_retrained", ascending=True)
    .reset_index(drop=True)
)

print("\nRetrained top-k results (sorted):")
print(retrained_top_k)

print("\nRetrained top-k results:")
print(retrained_top_k)



Retrained top-k results (sorted):
                        arch_name      act       opt    eta  MSE_original  \
0  3_Hidden_Layers (50, 100, 200)    LRELU      Adam  0.001      0.010843   
1       2_Hidden_Layers (50, 100)    LRELU      Adam  0.001      0.010531   
2       2_Hidden_Layers (50, 100)     RELU      Adam  0.001      0.010720   
3       2_Hidden_Layers (50, 100)    LRELU  RMS_prop  0.001      0.010440   
4       2_Hidden_Layers (50, 100)    LRELU       SGD  0.100      0.010539   
5             1_Hidden_Layer (50)  Sigmoid      Adam  0.100      0.010765   
6       2_Hidden_Layers (50, 100)     RELU  RMS_prop  0.001      0.010393   
7       2_Hidden_Layers (50, 100)     RELU       SGD  0.100      0.010535   
8       2_Hidden_Layers (50, 100)    LRELU  RMS_prop  0.010      0.010846   
9             1_Hidden_Layer (50)  Sigmoid  RMS_prop  0.010      0.010756   

   MSE_retrained  
0       0.010530  
1       0.010664  
2       0.010696  
3       0.011191  
4       0.011193  
5  